In [42]:
!pip install langchain
!pip install groq
!pip install pymupdf
!pip install langchain_groq
!pip install dotenv
!pip install sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
import langchain
from langchain_core.tools import tool

In [44]:
lease_contract = "C:\\Users\\urja tendolkar\\Desktop\\projects\\ClauseIQ\\training_docs\\lease_doc(contradict).pdf"
sell_contract = "C:\\Users\\urja tendolkar\\Desktop\\projects\\ClauseIQ\\training_docs\\sell_doc.pdf"

### Pdf content extraction

In [45]:
import re
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [46]:
import pymupdf as fitz

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc):

        # Extract page text
        words = page.get_text("words", sort = True)
        
        lines = []
        current_line = []
        current_y = None

        for word in words:
            x0, y0, x1, y1, text, block_no, line_no, word_no = word

            # New line if vertical position changes
            if current_y is None or abs(y0 - current_y) <= 3:
                current_line.append(word)
            else:
                current_line.sort(key=lambda w: w[0])
                lines.append(" ".join(w[4] for w in current_line))
                current_line = [word]

            current_y = y0

        if current_line:
            current_line.sort(key=lambda w: w[0])
            lines.append(" ".join(w[4] for w in current_line))

        pages.append({
            "page": page_num + 1,
            "text": "\n".join(lines)
        })


    doc.close()

    return pages
output = extract_pdf_text(lease_contract)
text = "\n".join(page["text"] for page in output)
# buy_text, buy_annotations = extract_pdf_text(sell_contract)
print(f"Extracted text: {text}") 
# print(f"Extracted text: {buy_text[:1000]}...") 
# Print first 100 bytes for preview


Extracted text: Final model draft agreed to by MoUD and DoLR
Note: -This is a model draft and may be customized according to individual
requirement.
LEASE DEED
This Lease deed is made and executed at (Name of place) on this ……………….day 20th of
…………, August ………… 2026
BETWEEN
……………………………….,s/od/o………………………………………., Mr. Rajesh Mehta Mr. Mahesh Mehta
r/o…………………….…………………… Kandivali, Mumbai, Maharashtra (hereinafter called the Lessor) of the one part.
AND
……………………………….,s/o Ms. Ananya Sharma d/o………………………………………., Mr. Rakesh Sharma
r/o…………………………………………… Hinjewadi, Pune, Maharashtra (hereinafter called the Lessee) of the other part.
The expression Lessor & Lessee shall mean and include the parties itself, their respective legal
heirs, executors, successors, administrators, legal representatives and assigns/nominees of their
respective part.
Whereas Lessor is an absolute owner and in possession of the property no. 502, ……………., Sunshine Residency
situated at
……………………………………………………measuring…………………….. And

In [47]:
from schema import Contract

In [48]:
print(Contract.schema_json(indent=2))  # Print the schema for the Contract model

{
  "$defs": {
    "Clause": {
      "description": "Represents a clause in a legal document.",
      "properties": {
        "title": {
          "description": "The title of the clause",
          "title": "Title",
          "type": "string"
        },
        "text": {
          "description": "The text content of the clause",
          "title": "Text",
          "type": "string"
        },
        "section": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "The section of the contract where the clause is located (if seperately specified)",
          "title": "Section"
        }
      },
      "required": [
        "title",
        "text"
      ],
      "title": "Clause",
      "type": "object"
    }
  },
  "description": "Represents a legal contract consisting of multiple clauses.",
  "properties": {
    "title": {
      "descri

C:\Users\urja tendolkar\AppData\Local\Temp\ipykernel_44328\50242175.py:1: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(Contract.schema_json(indent=2))  # Print the schema for the Contract model


In [49]:
import os
from langchain_groq import ChatGroq

client = ChatGroq(model="openai/gpt-oss-20b", 
                  api_key=os.getenv("GROQ_KEY"),
                
)


In [50]:
metadata_start = {
    "lease_deed":["property", "SCHEDULE OF THE LEASED PROPERTY:"],
    "sale_deed":"AGREEMENT TO SELL",
}
conditions_start = {
    "lease_deed": "NOW THIS LEASE DEED WITNESS AS UNDER:-",
    "sale_deed": "9. Conditions of Sale:",
    # add other document types later
}
def extract_clauses(contract, doc_type):
    start_phrase = conditions_start.get(doc_type)
    metadata_phrase = metadata_start.get(doc_type)
    if not start_phrase:
        raise ValueError(f"Unsupported document type: {doc_type}")

    parts = contract.split(start_phrase[0] if isinstance(start_phrase, list) else start_phrase)
    metadata = parts[0].strip() if len(parts) > 1 else ""
    clauses = parts[1].strip() if len(parts) > 1 else contract.strip()
    for phrase in metadata_phrase[1:] if isinstance(metadata_phrase, list) else [metadata_phrase]:
        if clauses.find(phrase) != -1:
            partition = clauses.split(phrase)
            clauses = partition[0].strip()
            metadata += "\n" + partition[1].strip() if len(partition) > 1 else ""

    return metadata, clauses

In [51]:
# @tool
# def find_related_clauses(clause: str, contract: str) -> str:
#     """Find other clauses in the contract that may interact with or modify the given clause."""
#     return "..."

# @tool
# def find_contradicting_clauses(clause: str, contract: str) -> str:
#     """Identify other clauses in the contract that contradict or conflict with the given clause."""
#     return "..."

In [52]:
metadata, clauses = extract_clauses(text, "lease_deed")


def cleanup(text):
    # Remove page number + footer
    text = re.sub(
        r'\s*\d+\s*Final model draft agreed to by MoUD and DoLR\n\s*',
        ' ',
        text
    )

    # Remove dotted / ellipsis placeholders
    text = re.sub(
        r'(?:\s*[.…]\s*){2,}',
        ' ',
        text
    )

    # Normalize whitespace and newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Clean metadata
metadata = cleanup(metadata)

# Split clauses by numbered sections
clause_list = re.split(r'\d+\.\s+', clauses)[1:]

# Clean each clause
clauses = {
    f"clause_{i}":cleanup(clause) for i, clause in enumerate(clause_list, start=1)
}

clauses
print("METADATA:")
print(metadata)

print("\nCLAUSES:")
print(clauses)

METADATA:
Final model draft agreed to by MoUD and DoLR Note: -This is a model draft and may be customized according to individual requirement. LEASE DEED This Lease deed is made and executed at (Name of place) on this day 20th of , August 2026 BETWEEN ,s/od/o , Mr. Rajesh Mehta Mr. Mahesh Mehta r/o Kandivali, Mumbai, Maharashtra (hereinafter called the Lessor) of the one part. AND ,s/o Ms. Ananya Sharma d/o , Mr. Rakesh Sharma r/o Hinjewadi, Pune, Maharashtra (hereinafter called the Lessee) of the other part. The expression Lessor & Lessee shall mean and include the parties itself, their respective legal heirs, executors, successors, administrators, legal representatives and assigns/nominees of their respective part. Whereas Lessor is an absolute owner and in possession of the property no. 502, , Sunshine Residency situated at measuring Andheri West, Mumbai, Maharashtra 1200 sq.ft (hereinafter referred to as the SAID PROPERTY). Revenue District Mumbai Suburban Sub-Registrar Office Andh

In [53]:
contract = {
    "contract_type": "lease_deed",
    "metadata": metadata,
    "clauses": clauses
}

#### Maintaining short term context leveraging vector embeddings and FAISS indexing

In [54]:
!pip install faiss-cpu sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
import faiss
import numpy as np

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4809.39it/s]


In [56]:
clause_ids = list(clauses.keys())
clause_texts = list(clauses.values())

embeddings = model.encode(
    clause_texts,
    normalize_embeddings=True
)

In [57]:
embeddings = np.array(embeddings, dtype="float32")

#### Indexing

In [58]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(index.ntotal)

19


In [61]:
retrieved_clauses = []

for idx, score in zip(indices[0], scores[0]):
    clause_id = clause_ids[idx]

    retrieved_clauses.append({
        "clause_id": clause_id,
        "score": float(score),
        "text": clauses[clause_id]
    })

In [62]:
retrieved_clauses

[{'clause_id': 'clause_17',
  'score': 0.5539824366569519,
  'text': 'That the said property can and shall be used only for the lawful purposes.'},
 {'clause_id': 'clause_16',
  'score': 0.353465735912323,
  'text': 'That the Lessor retains the right to inspect the said property, at reasonable time during the day, to know the condition of the building.'},
 {'clause_id': 'clause_15',
  'score': 0.3094085454940796,
  'text': 'That at the end of the lease period, the property shall be handed over to the lessor in good condition. The lessee shall have the right to refund the advance/premium paid subject to set-off of any claim of the lessor.'},
 {'clause_id': 'clause_13',
  'score': 0.2878960072994232,
  'text': 'That the lessee shall not make any structural changes in the said property or any portion thereof. However, minor changes/beautification may be made by the lessee with/ without the consent of the lessor in writing.'},
 {'clause_id': 'clause_11',
  'score': 0.2843685746192932,
  't

In [64]:
@tool
def classify_clause_risk(clause):
    """Classify a contract clause as LOW, MEDIUM, or HIGH risk."""

    prompt = """
    You are a legal document analysis assistant.

    Classify the following contract clause into one of three risk
    categories: LOW, MEDIUM, or HIGH.

    Criteria:

    LOW:
    The clause is standard, commonly accepted, and does not appear
    to impose significant obligations, restrictions, or unusual
    conditions.

    MEDIUM:
    The clause contains obligations, restrictions, ambiguity, or
    potentially burdensome conditions that may require attention
    or clarification.

    HIGH:
    The clause imposes significant obligations, restrictions,
    liabilities, penalties, or other conditions that could have
    serious implications for a party.

    Important:
    - Base the classification only on the provided clause.
    - Do not assume facts that are not present in the clause.
    - Do not declare a clause illegal or legally invalid.
    - Flag potential concerns that may warrant clarification or
      professional legal review.
    - Do not assess contradictions with other clauses here.
      Contradictions are handled separately.

    Respond in the following format:

    clause: <clause text>
    risk_category: <LOW, MEDIUM, or HIGH>
    justification: <brief explanation of the classification>
    """

    response = client.invoke(
        prompt + f"\n\nClause:\n{clause}"
    )

    return response.content

In [65]:
@tool
def summarize_clause(clause: str) -> str:
    """Explain a contract clause in simple language for a non-legal reader."""

    prompt = """
    You are a legal document analysis assistant.

    Your major function is to simplify complex legal language into
    clear, understandable explanations for non-legal readers.

    During your explanation, ensure that you:
    1. Use plain language and avoid unnecessary legal jargon.
    2. Provide context for legal terms that must be used.
    3. Do not provide legal advice or opinions.
    4. Do not assume facts that are not present in the clause.
    5. Keep the explanation concise and focused on the clause.
    6. Use examples or analogies only when they genuinely clarify
       a complex concept.
    7. Maintain the original meaning and intent of the clause.
    8. If the clause refers to another clause or document section,
       explain the reference in an understandable way.

    Contract clause:
    """

    response = client.invoke(
        prompt + f"\n\n{clause}"
    )

    return response.content

In [105]:
text_to_clause_id = {
    text.strip(): cid
    for cid, text in clauses.items()
}

In [119]:
import re

def normalize_clause(text):
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*/\s*', '/', text)
    return text


# clause_id → embedding/list index
clause_to_idx = {
    clause_id: idx
    for idx, clause_id in enumerate(clause_ids)
}

# normalized clause text → clause_id
normalized_clause_to_id = {
    normalize_clause(text): clause_id
    for clause_id, text in clauses.items()
}

In [120]:
@tool
def find_related_clauses(clause: str) -> str:
    """Find other clauses in the contract that may interact with or modify the given clause."""

    # 1. Find the clause ID from the text
    clause_id = normalized_clause_to_id.get(
        normalize_clause(clause)
    )

    if clause_id is None:
        return "Could not find the given clause in the current contract."

    # 2. Find its integer position in the embeddings
    clause_idx = clause_to_idx[clause_id]

    # 3. Get its embedding
    query_vector = embeddings[clause_idx].reshape(1, -1)

    # 4. Retrieve candidate clauses
    k = 6
    scores, indices = index.search(query_vector, k + 1)

    candidates = []

    for idx, score in zip(indices[0], scores[0]):
        retrieved_id = clause_ids[idx]

        if retrieved_id == clause_id:
            continue

        candidates.append({
            "clause_id": retrieved_id,
            "similarity": round(float(score), 4),
            "text": clauses[retrieved_id]
        })

        if len(candidates) == k:
            break

    # LLM validation...

    prompt = f"""
You are analyzing relationships between clauses in the SAME contract.

Primary clause:
{clause}

Candidate clauses:
{candidates}

Identify ONLY clauses that have a DIRECT contractual relationship
with the primary clause.

A candidate is related ONLY if:

1. The candidate explicitly refers to a right, obligation, condition,
   authority, restriction, or event established in the primary clause;

OR

2. The primary clause explicitly depends on, modifies, qualifies,
   or is affected by the candidate;

OR

3. Understanding one clause is necessary to correctly interpret
   or apply the other.

Do NOT mark a clause as related merely because:

- both clauses concern the same property;
- both clauses concern the same lease;
- one clause is logically possible because of the other;
- one party mentioned in one clause also appears in the other;
- the candidate would be relevant in a broader legal analysis;
- the candidate is a general consequence of ownership.

Do NOT infer legal relationships that are not explicitly supported
by the contract text.

Be conservative. It is completely acceptable to return NO related
clauses.

For every related clause, return:

Clause ID:
Reason:
Relationship type:

Return ONLY clauses that satisfy the direct-relationship criteria.
"""

    response = client.invoke(prompt)

    return response.content

In [121]:
from langchain.agents import create_agent

tools = [
    classify_clause_risk,
    summarize_clause,
    find_related_clauses
]

agent = create_agent(
    model=client,
    tools=tools,
    system_prompt="""
    You are ClauseIQ, a contract analysis assistant.

    Analyze contractual clauses for one or more of the following tasks:
    - Risk classification (LOW, MEDIUM, HIGH)
    - Simplified explanations for non-legal readers
    - Identification of related clauses within the contract

    Use the available tools when they are useful.
    Do not use all the tools for every clause; only use the tools that are relevant to the task at hand.
    Do not provide definitive legal advice.
    """
)

In [124]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""
            Give me related information I need to know about the following clause in this legal document.

            Clause: {clause_list[12]}
            Contract type: {contract['contract_type']}
            """
        }
    ]
})

In [125]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, SystemMessage

for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(msg.content)
    if isinstance(msg, ToolMessage):
        print(f"Tool output: {msg.content}")


Tool output: clause: That the lessee shall not make any structural changes in the said property or any portion thereof. However, minor changes/beautification may be made by the lessee with/without the consent of the lessor in writing.  
risk_category: LOW  
justification: The clause imposes a standard restriction on structural alterations, a common lease provision, and allows minor cosmetic changes with minimal conditions. No significant penalties, liabilities, or ambiguous obligations are introduced.

Tool output: **What the clause means**

- **No major changes:** The tenant (lessee) cannot do any big structural work—like adding a wall, moving a load‑bearing beam, or altering the building’s skeleton—on the property or any part of it.

- **Small changes are allowed:** The tenant may still make minor alterations or “beautify” the space (for example, painting, installing a new light fixture, or hanging a picture). These can be done with or without the landlord’s (lessor’s) written permi

In [114]:
result

{'messages': [HumanMessage(content='\n            Give me related information I need to know about the following clause in this legal document.\n\n            Clause: That the lessor is the sole/ joint/ co-owner of the said property being leased out and in\npossession of the said property mentioned in the Schedule below.\n\n            Contract type: lease_deed\n            ', additional_kwargs={}, response_metadata={}, id='c4f0aec2-ed86-450d-ba59-e333425b4371'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to provide related information the user needs to know about the clause. They want related info: risk classification, simplified explanation, related clauses. Use relevant tools. Likely use classify_clause_risk, summarize_clause, find_related_clauses. Provide risk classification, simple explanation, and related clauses. Let's call classify_clause_risk.", 'tool_calls': [{'id': 'fc_cbf814ab-e29c-4497-b8cc-9c078ae53255', 'function': {'arguments': '{"clause":"